# 📘 Capítulo 3 – Feature Engineering Temporal e Modelos de Machine Learning
**Autor:** Rodrigo Santana Ferreira  
**Disciplina:** Séries Temporais  

---
Este notebook contém os scripts Python apresentados no Capítulo 3, organizados por seção conforme o material da aula.

## 🔧 Instalação de Dependências

In [ ]:
# Instalação das bibliotecas necessárias
!pip install pandas numpy scikit-learn xgboost lightgbm matplotlib statsmodels

## 🏗️ Criação de Features Temporais Básicas e Avançadas
Pipeline completo de feature engineering: lags, rolling windows, diferenças, features cíclicas, encodings de calendário e decomposição STL (seção HANDS ON).

> **Nota:** Este código usa `vendas_diarias.csv` como exemplo. Substitua pelo seu dataset real.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import STL
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

# Carregar exemplo (substitua pelo seu dataset)
df = pd.read_csv('vendas_diarias.csv', parse_dates=['data'], index_col='data')
df = df.sort_index()

# Features básicas
df['lag_1'] = df['vendas'].shift(1)
df['lag_7'] = df['vendas'].shift(7)
df['diff_1'] = df['vendas'].diff(1)

# Rolling statistics
df['rolling_mean_7'] = df['vendas'].rolling(7).mean()
df['rolling_std_14'] = df['vendas'].rolling(14).std()

# Features cíclicas
df['dia_do_ano'] = df.index.dayofyear
df['sin_dia'] = np.sin(2 * np.pi * df['dia_do_ano'] / 365.25)
df['cos_dia'] = np.cos(2 * np.pi * df['dia_do_ano'] / 365.25)
df['dia_semana'] = df.index.dayofweek
df['sin_semana'] = np.sin(2 * np.pi * df['dia_semana'] / 7)
df['cos_semana'] = np.cos(2 * np.pi * df['dia_semana'] / 7)

# Features de calendário (exemplo simples)
df['fim_semana'] = df['dia_semana'].isin([5,6]).astype(int)
df['mes'] = df.index.month
df['trimestre'] = df.index.quarter

# Decomposição STL para extrair tendência e sazonalidade
stl = STL(df['vendas'], period=365)
res = stl.fit()
df['tendencia_stl'] = res.trend
df['sazonal_stl'] = res.seasonal

# Remover NaNs e preparar X/y
df = df.dropna()
X = df.drop('vendas', axis=1)
y = df['vendas']

# Validação temporal
tscv = TimeSeriesSplit(n_splits=5)
for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    model = GradientBoostingRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"MAE: {mean_absolute_error(y_test, pred):.2f}")

## 🏆 Comparação de Modelos de ML para Previsão de Séries Temporais
Comparação de RandomForest, GradientBoosting, XGBoost e LightGBM com validação cruzada temporal usando o dataset AirPassengers (seção HANDS ON).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import lightgbm as lgb

# Função para calcular MAPE (evita divisão por zero)
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero = y_true != 0
    return np.mean(np.abs((y_true[non_zero] - y_pred[non_zero]) / y_true[non_zero])) * 100

# Função para calcular MASE (Mean Absolute Scaled Error)
# Usa o erro naïve (previsão = valor do período anterior) como escala
def mean_absolute_scaled_error(y_true, y_pred, y_train):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    naive_errors = np.abs(np.diff(y_train))  # |y_t - y_{t-1}|
    scale = np.mean(naive_errors)
    if scale == 0:
        return np.nan
    return np.mean(np.abs(y_true - y_pred)) / scale

In [ ]:
# ────────────────────────────────────────────────
# 1. Carregar e preparar o dataset AirPassengers
# ────────────────────────────────────────────────
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url, parse_dates=['Month'], index_col='Month')
df.columns = ['Passengers']
df = df.asfreq('MS')  # Garante frequência mensal
print("Dataset shape:", df.shape)
print(df.head())

# Plot da série original
plt.figure(figsize=(12, 5))
plt.plot(df['Passengers'], label='Passengers')
plt.title('AirPassengers - Série Original')
plt.legend()
plt.show()

In [ ]:
# ────────────────────────────────────────────────
# 2. Feature Engineering Temporal (básico + cíclico)
# ────────────────────────────────────────────────
df['lag_1']   = df['Passengers'].shift(1)
df['lag_12']  = df['Passengers'].shift(12)  # sazonalidade anual
df['diff_1']  = df['Passengers'].diff(1)
df['rolling_mean_3']  = df['Passengers'].rolling(3).mean()
df['rolling_mean_12'] = df['Passengers'].rolling(12).mean()

# Features cíclicas (mês do ano)
df['month'] = df.index.month
df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)

# Remover NaNs gerados pelos shifts e rolling
df = df.dropna().copy()

# Definir X e y
X = df.drop('Passengers', axis=1)
y = df['Passengers']

print("Features criadas:", X.columns.tolist())
print("Shape final X/y:", X.shape, y.shape)

In [ ]:
# ────────────────────────────────────────────────
# 3. Definir modelos a comparar
# ────────────────────────────────────────────────
models = {
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
}

In [ ]:
# ────────────────────────────────────────────────
# 4. Validação cruzada temporal + cálculo de métricas
# ────────────────────────────────────────────────
tscv = TimeSeriesSplit(n_splits=5)  # 5 folds temporais
results = []

for name, model in models.items():
    mae_list, rmse_list, mape_list, mase_list = [], [], [], []
    
    for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        mae  = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mape = mean_absolute_percentage_error(y_test, y_pred)
        mase = mean_absolute_scaled_error(y_test, y_pred, y_train)
        
        mae_list.append(mae)
        rmse_list.append(rmse)
        mape_list.append(mape)
        mase_list.append(mase)
        
        print(f"Fold {fold} - {name}: MAE={mae:.2f}, RMSE={rmse:.2f}, MAPE={mape:.2f}%, MASE={mase:.3f}")
    
    # Médias do modelo
    results.append({
        'Modelo': name,
        'MAE_mean': np.mean(mae_list),
        'RMSE_mean': np.mean(rmse_list),
        'MAPE_mean': np.mean(mape_list),
        'MASE_mean': np.mean(mase_list)
    })

In [ ]:
# ────────────────────────────────────────────────
# 5. Resultados finais (tabela comparativa)
# ────────────────────────────────────────────────
results_df = pd.DataFrame(results).round(3)
print("\nComparação Final (médias dos 5 folds):")
print(results_df.sort_values('MASE_mean'))  # MASE é uma boa métrica escalada para séries temporais

# Visualização rápida
results_df.set_index('Modelo')[['MAE_mean', 'RMSE_mean', 'MAPE_mean', 'MASE_mean']].plot(kind='bar', figsize=(10,6))
plt.title('Comparação de Modelos - Métricas Médias')
plt.ylabel('Erro')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()